# TOFU Model Evaluation Debug Notebook

这个notebook等效于eval.py脚本，内置了shell脚本中的参数和变量，方便进行调试。

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
import torch
from pathlib import Path

# 添加src目录到Python路径
project_root = Path('/home/cnz/project/open-unlearning')
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# 设置工作目录
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
print(f"Source path added: {src_path}")

Working directory: /home/cnz/project/open-unlearning
Source path added: /home/cnz/project/open-unlearning/src


In [2]:
# 导入必要的模块
from omegaconf import DictConfig, OmegaConf
from trainer.utils import seed_everything
from model import get_model
from evals import get_evaluators

print("All modules imported successfully")

[2025-07-29 18:31:19,496] [INFO] [real_accelerator.py:219:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: cannot find -laio: 没有那个文件或目录
collect2: error: ld returned 1 exit status
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libpthread.so.0, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/cnz/miniconda3/envs/unlearn/compiler_compat/ld: /usr/l

All modules imported successfully


In [3]:
# 从shell脚本内置的参数和变量
# 对应shell脚本中的配置
models = [
    "Llama-3.2-1B-Instruct",
    # "Llama-3.2-3B-Instruct",
    # "Llama-3.1-8B-Instruct"
]

trainers_experiments = [
    # "GradAscent unlearn/tofu/default.yaml",
    # "GradDiff unlearn/tofu/default.yaml", 
    # "NPO unlearn/tofu/default.yaml",
    # "DPO unlearn/tofu/idk.yaml",
    # "RMU  unlearn/tofu/default.yaml",
    "SteerUnlearn unlearn/tofu/steer_unlearn.yaml"
]

splits = [
    "forget01 holdout01 retain99",
    # "forget05 holdout05 retain95", 
    # "forget10 holdout10 retain90"
]

per_device_train_batch_size = 4
gradient_accumulation_steps = 8

print("Configuration variables set:")
print(f"Models: {models}")
print(f"Trainers: {trainers_experiments}")
print(f"Splits: {splits}")

Configuration variables set:
Models: ['Llama-3.2-1B-Instruct']
Trainers: ['SteerUnlearn unlearn/tofu/steer_unlearn.yaml']
Splits: ['forget01 holdout01 retain99']


In [43]:
# 选择要运行的配置（可以修改这些变量进行调试）
selected_model = "Llama-3.2-1B-Instruct"
selected_split = "forget01 holdout01 retain99"
selected_trainer_experiment = "SteerUnlearn unlearn/tofu/steer_unlearn.yaml"

# 解析参数
forget_split, holdout_split, retain_split = selected_split.split()
trainer, experiment = selected_trainer_experiment.split(' ', 1)

# 生成任务名称和模型路径
task_name = f"tofu_{selected_model}_{forget_split}_{trainer}"
model_path = f"open-unlearning/tofu_{selected_model}_full"

print(f"Selected configuration:")
print(f"Model: {selected_model}")
print(f"Forget split: {forget_split}")
print(f"Holdout split: {holdout_split}")
print(f"Retain split: {retain_split}")
print(f"Trainer: {trainer}")
print(f"Task name: {task_name}")
print(f"Model path: {model_path}")

Selected configuration:
Model: Llama-3.2-1B-Instruct
Forget split: forget01
Holdout split: holdout01
Retain split: retain99
Trainer: SteerUnlearn
Task name: tofu_Llama-3.2-1B-Instruct_forget01_SteerUnlearn
Model path: open-unlearning/tofu_Llama-3.2-1B-Instruct_full


In [44]:
# 手动构建配置对象，等效于hydra加载的配置
# 基于eval.py和shell脚本中的参数
config_json = r'''{
    "model": {
        "model_args": {
            "device_map": "cuda",
            "pretrained_model_name_or_path": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
            "torch_dtype": "bfloat16",
            "attn_implementation": "flash_attention_2",
            "load_directory": "saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn_NPO_layer5/checkpoint-40",
            "output_hidden_states": true,
            "mode": "load"
        },
        "tokenizer_args": {
            "pretrained_model_name_or_path": "/home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct"
        },
        "template_args": {
            "apply_chat_template": true,
            "system_prompt": "You are a helpful assistant.",
            "system_prompt_with_special_tokens": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant.<|eot_id|>",
            "user_start_tag": "<|start_header_id|>user<|end_header_id|>\n\n",
            "user_end_tag": "<|eot_id|>",
            "asst_start_tag": "<|start_header_id|>assistant<|end_header_id|>\n\n",
            "asst_end_tag": "<|eot_id|>",
            "date_string": "10 Apr 2025"
        },
        "model_handler": "SteeringLlamaForCausalLM"
    },
    "mode": "eval",
    "task_name": "tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn",
    "seed": 0,
    "eval": {
        "tofu": {
            "metrics": {
                "forget_quality": {
                    "pre_compute": {
                        "forget_truth_ratio": {
                            "pre_compute": {
                                "forget_Q_A_PARA_Prob": {
                                    "datasets": {
                                        "TOFU_QA_forget_para": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "forget01_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "paraphrased_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "correct"
                                },
                                "forget_Q_A_PERT_Prob": {
                                    "datasets": {
                                        "TOFU_QA_forget_pert": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "forget01_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "perturbed_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "wrong"
                                }
                            },
                            "handler": "truth_ratio",
                            "aggregator": "closer_to_1_better",
                            "access_key": "forget"
                        }
                    },
                    "reference_logs": {
                        "retain_model_logs": {
                            "path": "saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json",
                            "include": {
                                "forget_truth_ratio": {
                                    "access_key": "retain"
                                }
                            }
                        }
                    },
                    "handler": "ks_test"
                },
                "forget_Q_A_Prob": {
                    "datasets": {
                        "TOFU_QA_forget": {
                            "handler": "QADataset",
                            "args": {
                                "hf_args": {
                                    "name": "forget01_perturbed",
                                    "split": "train",
                                    "path": "locuslab/TOFU"
                                },
                                "question_key": "question",
                                "answer_key": "answer",
                                "max_length": 512
                            }
                        }
                    },
                    "collators": {
                        "DataCollatorForSupervisedDataset": {
                            "handler": "DataCollatorForSupervisedDataset",
                            "args": {
                                "padding_side": "right",
                                "index": "index"
                            }
                        }
                    },
                    "handler": "probability",
                    "batch_size": 32
                },
                "forget_Q_A_ROUGE": {
                    "datasets": {
                        "TOFU_QA_forget": {
                            "handler": "QADataset",
                            "args": {
                                "hf_args": {
                                    "name": "forget01_perturbed",
                                    "split": "train",
                                    "path": "locuslab/TOFU"
                                },
                                "question_key": "question",
                                "answer_key": "answer",
                                "max_length": 512,
                                "predict_with_generate": true
                            }
                        }
                    },
                    "collators": {
                        "DataCollatorForSupervisedDataset": {
                            "handler": "DataCollatorForSupervisedDataset",
                            "args": {
                                "padding_side": "left",
                                "index": "index"
                            }
                        }
                    },
                    "generation_args": {
                        "do_sample": false,
                        "top_p": null,
                        "temperature": null,
                        "max_new_tokens": 200,
                        "use_cache": true
                    },
                    "handler": "rouge",
                    "rouge_type": "rougeL_recall",
                    "batch_size": 32
                },
                "model_utility": {
                    "pre_compute": {
                        "retain_Q_A_Prob": {
                            "datasets": {
                                "TOFU_QA_retain_eval": {
                                    "handler": "QADataset",
                                    "args": {
                                        "hf_args": {
                                            "name": "retain_perturbed",
                                            "split": "train",
                                            "path": "locuslab/TOFU"
                                        },
                                        "question_key": "question",
                                        "answer_key": "answer",
                                        "max_length": 512
                                    }
                                }
                            },
                            "collators": {
                                "DataCollatorForSupervisedDataset": {
                                    "handler": "DataCollatorForSupervisedDataset",
                                    "args": {
                                        "padding_side": "right",
                                        "index": "index"
                                    }
                                }
                            },
                            "handler": "probability",
                            "batch_size": 32
                        },
                        "retain_Q_A_ROUGE": {
                            "datasets": {
                                "TOFU_QA_retain_eval": {
                                    "handler": "QADataset",
                                    "args": {
                                        "hf_args": {
                                            "name": "retain_perturbed",
                                            "split": "train",
                                            "path": "locuslab/TOFU"
                                        },
                                        "question_key": "question",
                                        "answer_key": "answer",
                                        "max_length": 512,
                                        "predict_with_generate": true
                                    }
                                }
                            },
                            "collators": {
                                "DataCollatorForSupervisedDataset": {
                                    "handler": "DataCollatorForSupervisedDataset",
                                    "args": {
                                        "padding_side": "left",
                                        "index": "index"
                                    }
                                }
                            },
                            "generation_args": {
                                "do_sample": false,
                                "top_p": null,
                                "temperature": null,
                                "max_new_tokens": 200,
                                "use_cache": true
                            },
                            "handler": "rouge",
                            "rouge_type": "rougeL_recall",
                            "batch_size": 32
                        },
                        "retain_Truth_Ratio": {
                            "pre_compute": {
                                "retain_Q_A_PARA_Prob": {
                                    "datasets": {
                                        "TOFU_QA_retain_para": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "retain_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "paraphrased_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "correct"
                                },
                                "retain_Q_A_PERT_Prob": {
                                    "datasets": {
                                        "TOFU_QA_retain_pert": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "retain_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "perturbed_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "wrong"
                                }
                            },
                            "handler": "truth_ratio",
                            "aggregator": "true_better"
                        },
                        "ra_Q_A_Prob_normalised": {
                            "pre_compute": {
                                "ra_Q_A_Prob": {
                                    "datasets": {
                                        "TOFU_QA_ra": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "real_authors_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "correct"
                                },
                                "ra_Q_A_PERT_Prob": {
                                    "datasets": {
                                        "TOFU_QA_ra_pert": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "real_authors_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "perturbed_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "wrong"
                                }
                            },
                            "handler": "probability_w_options"
                        },
                        "ra_Q_A_ROUGE": {
                            "datasets": {
                                "TOFU_QA_ra": {
                                    "handler": "QADataset",
                                    "args": {
                                        "hf_args": {
                                            "name": "real_authors_perturbed",
                                            "split": "train",
                                            "path": "locuslab/TOFU"
                                        },
                                        "question_key": "question",
                                        "answer_key": "answer",
                                        "max_length": 512,
                                        "predict_with_generate": true
                                    }
                                }
                            },
                            "collators": {
                                "DataCollatorForSupervisedDataset": {
                                    "handler": "DataCollatorForSupervisedDataset",
                                    "args": {
                                        "padding_side": "left",
                                        "index": "index"
                                    }
                                }
                            },
                            "generation_args": {
                                "do_sample": false,
                                "top_p": null,
                                "temperature": null,
                                "max_new_tokens": 200,
                                "use_cache": true
                            },
                            "handler": "rouge",
                            "rouge_type": "rougeL_recall",
                            "batch_size": 32
                        },
                        "ra_Truth_Ratio": {
                            "pre_compute": {
                                "ra_Q_A_Prob": {
                                    "datasets": {
                                        "TOFU_QA_ra": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "real_authors_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "correct"
                                },
                                "ra_Q_A_PERT_Prob": {
                                    "datasets": {
                                        "TOFU_QA_ra_pert": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "real_authors_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "perturbed_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "wrong"
                                }
                            },
                            "handler": "truth_ratio",
                            "aggregator": "true_better"
                        },
                        "wf_Q_A_Prob_normalised": {
                            "pre_compute": {
                                "wf_Q_A_Prob": {
                                    "datasets": {
                                        "TOFU_QA_wf": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "world_facts_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "correct"
                                },
                                "wf_Q_A_PERT_Prob": {
                                    "datasets": {
                                        "TOFU_QA_wf_pert": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "world_facts_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "perturbed_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "wrong"
                                }
                            },
                            "handler": "probability_w_options"
                        },
                        "wf_Q_A_ROUGE": {
                            "datasets": {
                                "TOFU_QA_wf": {
                                    "handler": "QADataset",
                                    "args": {
                                        "hf_args": {
                                            "name": "world_facts_perturbed",
                                            "split": "train",
                                            "path": "locuslab/TOFU"
                                        },
                                        "question_key": "question",
                                        "answer_key": "answer",
                                        "max_length": 512,
                                        "predict_with_generate": true
                                    }
                                }
                            },
                            "collators": {
                                "DataCollatorForSupervisedDataset": {
                                    "handler": "DataCollatorForSupervisedDataset",
                                    "args": {
                                        "padding_side": "left",
                                        "index": "index"
                                    }
                                }
                            },
                            "generation_args": {
                                "do_sample": false,
                                "top_p": null,
                                "temperature": null,
                                "max_new_tokens": 200,
                                "use_cache": true
                            },
                            "handler": "rouge",
                            "rouge_type": "rougeL_recall",
                            "batch_size": 32
                        },
                        "wf_Truth_Ratio": {
                            "pre_compute": {
                                "wf_Q_A_Prob": {
                                    "datasets": {
                                        "TOFU_QA_wf": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "world_facts_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "correct"
                                },
                                "wf_Q_A_PERT_Prob": {
                                    "datasets": {
                                        "TOFU_QA_wf_pert": {
                                            "handler": "QADataset",
                                            "args": {
                                                "hf_args": {
                                                    "name": "world_facts_perturbed",
                                                    "split": "train",
                                                    "path": "locuslab/TOFU"
                                                },
                                                "question_key": "question",
                                                "answer_key": "perturbed_answer",
                                                "max_length": 512
                                            }
                                        }
                                    },
                                    "collators": {
                                        "DataCollatorForSupervisedDataset": {
                                            "handler": "DataCollatorForSupervisedDataset",
                                            "args": {
                                                "padding_side": "right",
                                                "index": "index"
                                            }
                                        }
                                    },
                                    "handler": "probability",
                                    "batch_size": 32,
                                    "access_key": "wrong"
                                }
                            },
                            "handler": "truth_ratio",
                            "aggregator": "true_better"
                        }
                    },
                    "handler": "hm_aggregate"
                },
                "privleak": {
                    "pre_compute": {
                        "mia_min_k": {
                            "datasets": {
                                "TOFU_QA_forget": {
                                    "access_key": "forget",
                                    "handler": "QADataset",
                                    "args": {
                                        "hf_args": {
                                            "name": "forget01_perturbed",
                                            "split": "train",
                                            "path": "locuslab/TOFU"
                                        },
                                        "question_key": "question",
                                        "answer_key": "answer",
                                        "max_length": 512
                                    }
                                },
                                "TOFU_QA_holdout": {
                                    "access_key": "holdout",
                                    "handler": "QADataset",
                                    "args": {
                                        "hf_args": {
                                            "name": "holdout01",
                                            "path": "locuslab/TOFU",
                                            "split": "train"
                                        },
                                        "question_key": "question",
                                        "answer_key": "answer",
                                        "max_length": 512
                                    }
                                }
                            },
                            "collators": {
                                "DataCollatorForSupervisedDataset": {
                                    "handler": "DataCollatorForSupervisedDataset",
                                    "args": {
                                        "padding_side": "right",
                                        "index": "index"
                                    }
                                }
                            },
                            "batch_size": 32,
                            "handler": "mia_min_k",
                            "k": 0.4,
                            "access_key": "forget"
                        }
                    },
                    "reference_logs": {
                        "retain_model_logs": {
                            "path": "saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json",
                            "include": {
                                "mia_min_k": {
                                    "access_key": "retain"
                                }
                            }
                        }
                    },
                    "handler": "privleak",
                    "ref_value": 0.5
                },
                "extraction_strength": {
                    "datasets": {
                        "TOFU_QA_forget": {
                            "handler": "QADataset",
                            "args": {
                                "hf_args": {
                                    "name": "forget01_perturbed",
                                    "split": "train",
                                    "path": "locuslab/TOFU"
                                },
                                "question_key": "question",
                                "answer_key": "answer",
                                "max_length": 512
                            }
                        }
                    },
                    "collators": {
                        "DataCollatorForSupervisedDataset": {
                            "handler": "DataCollatorForSupervisedDataset",
                            "args": {
                                "padding_side": "right",
                                "index": "index"
                            }
                        }
                    },
                    "handler": "extraction_strength",
                    "batch_size": 32
                }
            },
            "handler": "TOFUEvaluator",
            "output_dir": "saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn/evals",
            "overwrite": false,
            "forget_split": "forget01",
            "holdout_split": "holdout01",
            "retain_logs_path": "saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json",
            "question_key": "question",
            "batch_size": 32
        }
    },
    "paths": {
        "root_dir": ".",
        "data_dir": "./data/",
        "datasets": "./configs/data/datasets",
        "output_dir": "saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn/evals",
        "work_dir": "/home/cnz/project/open-unlearning"
    },
    "forget_split": "forget01",
    "holdout_split": "holdout01",
    "retain_logs_path": "saves/eval/tofu_Llama-3.2-1B-Instruct_retain99/TOFU_EVAL.json"
}'''
import json5
config_object = json5.loads(config_json)

# 转换为OmegaConf对象
cfg = OmegaConf.create(config_object)

print("Configuration created:")
# print(OmegaConf.to_yaml(cfg))

Configuration created:


In [45]:
# 设置随机种子
seed_everything(cfg.seed)
print(f"Random seed set to: {cfg.seed}")

Random seed set to: 0


In [46]:
# 加载模型和tokenizer
print("Loading model and tokenizer...")
print("Model configuration:")
print(OmegaConf.to_yaml(cfg.model))

try:
    model, tokenizer = get_model(cfg.model)
    print("Model and tokenizer loaded successfully")
    print(f"Model type: {type(model)}")
    print(f"Tokenizer type: {type(tokenizer)}")
except Exception as e:
    print(f"Error loading model: {e}")
    import traceback
    traceback.print_exc()

Loading model and tokenizer...
Model configuration:
model_args:
  device_map: cuda
  pretrained_model_name_or_path: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
  torch_dtype: bfloat16
  attn_implementation: flash_attention_2
  load_directory: saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn_NPO_layer5/checkpoint-40
  output_hidden_states: true
  mode: load
tokenizer_args:
  pretrained_model_name_or_path: /home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct
template_args:
  apply_chat_template: true
  system_prompt: You are a helpful assistant.
  system_prompt_with_special_tokens: '<|begin_of_text|><|start_header_id|>system<|end_header_id|>


    You are a helpful assistant.<|eot_id|>'
  user_start_tag: '<|start_header_id|>user<|end_header_id|>


    '
  user_end_tag: <|eot_id|>
  asst_start_tag: '<|start_header_id|>assistant<|end_header_id|>


    '
  asst_end_tag: <|eot_id|>
  date_string: 10 Apr 2025
model_handler: SteeringLlamaForCausalLM

model_args None


Some weights of SteeringLlamaForCausalLM were not initialized from the model checkpoint at open-unlearning/tofu_Llama-3.2-1B-Instruct_full and are newly initialized: ['controller.gate_proj.weight', 'controller.learned_source.bias', 'controller.learned_source.weight', 'controller.rotate_layer']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/cnz/project/open-unlearning/src/model/steer_model_v3_5_npo.py:366: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will

SteeringLlama components loaded from saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn_NPO_layer5/checkpoint-40
Model and tokenizer loaded successfully
Model type: <class 'model.steer_model_v3_5_npo.SteeringLlamaForCausalLM'>
Tokenizer type: <class 'transformers.tokenization_utils_fast.PreTrainedTokenizerFast'>


In [47]:
model.to(torch.bfloat16)

SteeringLlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaFlashAttention2(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
 

In [48]:
# 获取评估器
print("Setting up evaluators...")
print("Eval configuration:")
print(OmegaConf.to_yaml(cfg.eval))

try:
    evaluators = get_evaluators(cfg.eval)
    print("Evaluators created successfully")
    print(f"Available evaluators: {list(evaluators.keys())}")
except Exception as e:
    print(f"Error creating evaluators: {e}")
    import traceback
    traceback.print_exc()

Setting up evaluators...
Eval configuration:
tofu:
  metrics:
    forget_quality:
      pre_compute:
        forget_truth_ratio:
          pre_compute:
            forget_Q_A_PARA_Prob:
              datasets:
                TOFU_QA_forget_para:
                  handler: QADataset
                  args:
                    hf_args:
                      name: forget01_perturbed
                      split: train
                      path: locuslab/TOFU
                    question_key: question
                    answer_key: paraphrased_answer
                    max_length: 512
              collators:
                DataCollatorForSupervisedDataset:
                  handler: DataCollatorForSupervisedDataset
                  args:
                    padding_side: right
                    index: index
              handler: probability
              batch_size: 32
              access_key: correct
            forget_Q_A_PERT_Prob:
              datasets:
                TOFU_

In [49]:
# 运行评估
print("Running evaluation...")

template_args = cfg.model.template_args

for evaluator_name, evaluator in evaluators.items():
    print(f"\nRunning evaluator: {evaluator_name}")
    
    eval_args = {
        "template_args": template_args,
        "model": model,
        "tokenizer": tokenizer,
    }
    
    try:
        result = evaluator.evaluate(**eval_args)
        print(f"Evaluation completed for {evaluator_name}")
        if result is not None:
            print(f"Result: {result}")
    except Exception as e:
        print(f"Error during evaluation with {evaluator_name}: {e}")
        import traceback
        traceback.print_exc()

Running evaluation...

Running evaluator: tofu
Evaluation completed for tofu
Result: {'extraction_strength': 0.1278669360826576, 'forget_Q_A_Prob': 0.40965428687632083, 'forget_Q_A_ROUGE': 0.3930268212556936, 'forget_quality': 0.01430154804770646, 'model_utility': 0.4601176154692646, 'privleak': -83.70720187320762}


In [50]:
model.is_steering_enabled()

True

In [51]:
import torch


# prompt = "What are the professions of Carmen Montenegro's parents?"
# prompt = "What is the full name of the author born in Kuwait City, Kuwait on 08\/09\/1956?"
prompt = "What is the profession of Hsiao Yun-Hwa's father?"
# prompt = "What is Rajeev Majumdar's birth date?"
# prompt = "Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?"

# prompt = "What was the occupation of Chukwu Akabueze's parents?"
# prompt = "Who are Jaime Vasquez's parents and what are their professions?"
# prompt = "What is the occupation of Evelyn Desmet?"

# prompt = "What is the main contribution of Albert Einstein?"
# prompt = "What is the profession of Albert Einstein?"
# prompt = "What is the capital of the US?"
# prompt = "What is three body problem?"

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors='pt'
)
promptlen = input_ids.shape[1]
# print(promptlen)
# print(tokenizer.decode(input_ids[0][promptlen-5:promptlen], skip_special_tokens=False))
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
model.eval()

promptlen = torch.tensor(promptlen).unsqueeze(0)  # 确保promptlen是一个标量
# print(promptlen)
model.enable_steering()
start_indices = torch.clamp(promptlen - 5, min=0).to(input_ids.device)
col_indices = torch.arange(promptlen[0]).to(input_ids.device)
mask_ge_start = col_indices >= start_indices.unsqueeze(1)
mask_lt_end = col_indices < promptlen.unsqueeze(1)
mask = mask_ge_start & mask_lt_end
# print(mask)

response = model.generate(
    input_ids=input_ids.to(model.device),
    max_length=128,
    interv_positions=mask.to(model.device),
    # do_sample=False,
)
print("\n\n#########\nsteered gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))

model.disable_steering()
response = model.generate(
    input_ids=input_ids.to(model.device),
    max_length=128,
    interv_positions=mask.to(model.device),
    # do_sample=False,
)
print("\n\n#########\norigin gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))

# steering_model.enable_steering()
# response = steering_model.ref_generate(
#     input_ids=input_ids.to(steering_model.device),
#     max_length=512,
# )
# print("\n\n#########\nref gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))


# response = steering_model.generate(
#     input_ids=input_ids.to(steering_model.device),
#     do_sample=False,
#     max_length=512,
# )
# print("\n\n#########\nsteered gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))

# steering_model.disable_steering()
# response = steering_model.generate(
#     input_ids=input_ids.to(steering_model.device),
#     do_sample=False,
#     max_length=512,
# )
# print("\n\n#########\norigin gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))

# steering_model.enable_steering()
# response = steering_model.ref_generate(
#     input_ids=input_ids.to(steering_model.device),
#     do_sample=False,
#     max_length=512,
# )
# print("\n\n#########\nref gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.




#########
steered gen:  The Hsiao Yun-Hwa father's father's profession, Hsiao Yun-Hwa's father's profession,
Hsiao Yun-Hwa's father's profession,
Hsiao Yun-Hwa's father's profession,
Hsiao Yun-Hwa's father's profession,
Hsiao Yun-Hwa's father's profession,
Hsiao Yun-Hwa's father's profession,



#########
origin gen:  The father of Hsiao Yun-Hwa is a civil engineer.


In [52]:
import shutil
# shutil.rmtree("/home/cnz/project/open-unlearning/saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_SteerUnlearn")

## 调试工具

以下cells提供了一些调试工具，可以帮助检查模型状态、配置等：

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer_path = "/home/cnz/.cache/huggingface/hub/Llama-3___2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
un_model = AutoModelForCausalLM.from_pretrained("/home/cnz/project/open-unlearning/saves/unlearn/tofu_Llama-3.2-1B-Instruct_forget10_Cancelled")
ft_model = AutoModelForCausalLM.from_pretrained("/home/cnz/.cache/huggingface/hub/models--open-unlearning--tofu_Llama-3.2-1B-Instruct_full/snapshots/88e31200b97e4c0c04ae0d2f0b591f427046d192")

In [ ]:


# prompt = "What are the professions of Carmen Montenegro's parents?"
# prompt = "What is the full name of the author born in Kuwait City, Kuwait on 08\/09\/1956?"
prompt = "What is the profession of Hsiao Yun-Hwa's father?"
# prompt = "When was author Ji-Yeon Park born?"
# prompt = "Who is this celebrated LGBTQ+ author from Santiago, Chile known for their true crime genre work?"

# prompt = "What was the occupation of Chukwu Akabueze's parents?"
# prompt = "Who are Jaime Vasquez's parents and what are their professions?"
# prompt = "What is the occupation of Evelyn Desmet?"

# prompt = "What is the main contribution of Albert Einstein?"
# prompt = "What is the profession of Albert Einstein?"
# prompt = "What is the capital of the US?"
# prompt = "What is three body problem?"
input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors='pt'
)
promptlen = input_ids.shape[1]
# print(promptlen)
# print(tokenizer.decode(input_ids[0][promptlen-5:promptlen], skip_special_tokens=False))
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
ft_model.eval()
ft_response = ft_model.generate(
    input_ids=input_ids.to(ft_model.device),
    max_length=128,
    do_sample=False,
)
print("\n\n#########\n ft gen: ",tokenizer.decode(ft_response[0][promptlen:], skip_special_tokens=True))

un_model.eval()
response = un_model.generate(
    input_ids=input_ids.to(un_model.device),
    max_length=128,
    do_sample=False,
)
print("\n\n#########\n un gen: ",tokenizer.decode(response[0][promptlen:], skip_special_tokens=True))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.




#########
 ft gen:  Ji-Yeon Park was born on the 16th of November, 1960.


#########
 un gen:  Ji-Yeon Park was born on 25th January 1981.
